In [1]:
import pandas as pd 
import mysql.connector

In [2]:
#get the engine to read excel file
!pip install openpyxl

In [3]:
# extract data into dataframes
#CSV files
orders_df = pd.read_csv("C:\\Users\\nutan\\OneDrive\\Desktop\\ETL_project\\orders_data.csv")
orders_df.head()

,UserID,OrderDate,TotalAmount,Email
0,8,InvalidDate,NaN,user8@example.com
1,4,8/2/2024 0:00,365.00,user4@example.com
2,1,8/3/2024 0:00,446.59,user1@example.com
3,8,8/4/2024 0:00,866.02,user8@example.com
4,4,8/5/2024 0:00,385.23,user4@example.com


In [4]:
users_df = pd.read_csv("C:\\Users\\nutan\\OneDrive\\Desktop\\ETL_project\\users_data.csv")
users_df.head()

,UserName,Email,Address,Password
0,User A,invalid email,1 Example St,password1
1,User B,user1@example.com,2 Example St,password2
2,User C,user2@example.com,3 Example St,password3
3,User D,user3@example.com,4 Example St,password4
4,User E,user4@example.com,5 Example St,password5


In [5]:
products_df = pd.read_csv("C:\\Users\\nutan\\OneDrive\\Desktop\\ETL_project\\products_data.csv")
products_df.head()

,ProductName,Description,Price,StockQuantity,Category
0,Contraption A,NaN,free,unknown,NaN
1,Tool B,A high-quality tool.,230.23,160,Apparatus
2,Machine C,A high-quality apparatus.,56.75,96,Apparatus
3,Device D,NaN,191.7,472,Widgets
4,Contraption E,A high-quality gadget.,337.73,233,Instruments


In [6]:
#EXCEL files
#engine - bridge which gets data from excel file
payments_df = pd.read_excel("C:\\Users\\nutan\\OneDrive\\Desktop\\ETL_project\\payments_data.xlsx",engine = "openpyxl")
payments_df.head()

,OrderID,PaymentMethod,PaymentDate,Amount
0,1001,InvalidMethod,NaT,NaN
1,1002,Credit Card,2024-08-02,753.48
2,1003,Credit Card,2024-08-03,978.27
3,1004,InvalidMethod,2024-08-04,564.67
4,1005,Credit Card,2024-08-05,390.66


In [7]:
reviews_df = pd.read_excel("C:\\Users\\nutan\\OneDrive\\Desktop\\ETL_project\\reviews_data.xlsx",engine = "openpyxl")
reviews_df.head()

,ProductID,UserID,Rating,ReviewText
0,27,3,InvalidRating,Great product!
1,20,4,2,Great product!
2,24,2,5,Great product!
3,12,1,5,Great product!
4,50,7,5,Great product!


In [8]:
print("Extract Successful")
orders_df.head()

Extract Successful


,UserID,OrderDate,TotalAmount,Email
0,8,InvalidDate,NaN,user8@example.com
1,4,8/2/2024 0:00,365.00,user4@example.com
2,1,8/3/2024 0:00,446.59,user1@example.com
3,8,8/4/2024 0:00,866.02,user8@example.com
4,4,8/5/2024 0:00,385.23,user4@example.com


In [9]:
#----------Transform data - cleaning

#fill missing values in each dataframe

#--------PRODUCTS TABLE
products_df = products_df.fillna({
    'Description': "No Description",
    'Category' : "Miscellaneous"
})

products_df['Price'] = pd.to_numeric(products_df['Price'], errors = 'coerce').fillna(0.0).round(2)
#to_numeric - changes datatype to number
# errors = 'coerce' - will show as NaN where value is not numeric
# NaN = Null, fillna - will replace the NaN with 0
#round - will set it as float value 
products_df['StockQuantity'] = pd.to_numeric(products_df['StockQuantity'], errors = 'coerce').fillna(0).round(0)

print(products_df)


      ProductName                  Description   Price  StockQuantity  \
0   Contraption A               No Description    0.00            0.0   
1          Tool B         A high-quality tool.  230.23          160.0   
2       Machine C    A high-quality apparatus.   56.75           96.0   
3        Device D               No Description  191.70          472.0   
4   Contraption E       A high-quality gadget.  337.73          233.0   
5     Appliance F    A high-quality appliance.  336.30          180.0   
6         Gizmo G               No Description  299.74          113.0   
7   Contraption H       A high-quality gadget.  144.61          318.0   
8       Machine I    A high-quality appliance.  285.01            0.0   
9        Device J               No Description  197.63          497.0   
10         Tool K         A high-quality tool.    0.00          442.0   
11      Machine L      A high-quality machine.  486.14           52.0   
12      Machine M               No Description  425

In [10]:
#-------ORDERS TABLE

orders_df = orders_df.fillna({
    'TotalAmount': 0.0
})

#orders_df['OrderDate'] = orders_df['OrderDate'].str.strip() # remove white spaces from string Invalid date)
orders_df['OrderDate'] = pd.to_datetime(orders_df['OrderDate'], errors = 'coerce')
orders_df['OrderDate'] = orders_df['OrderDate'].fillna(pd.Timestamp('1970-01-01'))
orders_df['OrderDate'] = orders_df['OrderDate'].dt.strftime('%Y-%m-%d')
#strftime() - converts date to given date format


# in dataframe add a new column which will have order ids's in range 1001 to total_records
# create original order id for mapping
orders_df["OldOrderID"] = range(1001, 1001 + len(orders_df))
#range - function which takes start value, end value and step value(1). fill the new column with values
print(orders_df)

    UserID   OrderDate  TotalAmount              Email  OldOrderID
0        8  1970-01-01         0.00  user8@example.com        1001
1        4  2024-08-02       365.00  user4@example.com        1002
2        1  2024-08-03       446.59  user1@example.com        1003
3        8  2024-08-04       866.02  user8@example.com        1004
4        4  2024-08-05       385.23  user4@example.com        1005
5        6  2024-08-06       252.54  user6@example.com        1006
6        8  2024-08-07       601.12  user8@example.com        1007
7        4  1970-01-01       942.54  user4@example.com        1008
8        3  2024-08-09       726.43  user3@example.com        1009
9        9  2024-08-10       613.06  user9@example.com        1010
10       3  2024-08-11         0.00  user3@example.com        1011
11       9  2024-08-12       187.46  user9@example.com        1012
12       2  2024-08-13       653.51  user2@example.com        1013
13       2  2024-08-14       991.05  user2@example.com        

C:\Users\nutan\AppData\Local\Temp\ipykernel_17276\3732985486.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders_df['OrderDate'] = pd.to_datetime(orders_df['OrderDate'], errors = 'coerce')


In [11]:
#--------USER TABLE

#function to generate random emailid
def generate_unique_email(index):
    return f'unknown{index}@example.com'
#index - index number assigned to each row in df    
users_df['Email'] = users_df.apply(lambda row: row['Email'] if '@' in row['Email'] and '.' in row['Email'] else generate_unique_email(row.name) ,axis = 1)
#row.name - gets the row index number
#axis = 1 will check each row, axis = 0 - checks each column
#apply() - is a higher order function - takes another function as argument
#will apply the given function on each row
print(users_df)

  UserName                 Email        Address    Password
0   User A  unknown0@example.com   1 Example St   password1
1   User B     user1@example.com   2 Example St   password2
2   User C     user2@example.com   3 Example St   password3
3   User D     user3@example.com   4 Example St   password4
4   User E     user4@example.com   5 Example St   password5
5   User F  unknown5@example.com   6 Example St   password6
6   User G     user6@example.com   7 Example St   password7
7   User H     user7@example.com   8 Example St   password8
8   User I     user8@example.com   9 Example St   password9
9   User J     user9@example.com  10 Example St  password10


In [12]:
#------PAYMENTS TABLE
payments_df = payments_df.fillna(
    {
        'Amount':0.0
    }
)
payments_df['PaymentDate'] = pd.to_datetime(payments_df['PaymentDate'], errors = 'coerce', format = '%Y-%m-%d').fillna(pd.Timestamp('1970-01-01'))
payments_df['PaymentDate'] = payments_df['PaymentDate'].dt.strftime('%Y-%m-%d')

payments_df['PaymentMethod'] = payments_df['PaymentMethod'].replace('InvalidMethod','Unknown')
print(payments_df)

    OrderID PaymentMethod PaymentDate  Amount
0      1001       Unknown  1970-01-01    0.00
1      1002   Credit Card  2024-08-02  753.48
2      1003   Credit Card  2024-08-03  978.27
3      1004       Unknown  2024-08-04  564.67
4      1005   Credit Card  2024-08-05  390.66
5      1006   Credit Card  1970-01-01  815.67
6      1007       Unknown  2024-08-07  343.75
7      1008   Credit Card  2024-08-08  495.07
8      1009   Credit Card  2024-08-09  170.61
9      1010       Unknown  2024-08-10  122.82
10     1011   Credit Card  1970-01-01    0.00
11     1012   Credit Card  2024-08-12  966.38
12     1013       Unknown  2024-08-13  852.38
13     1014   Credit Card  2024-08-14  726.38
14     1015   Credit Card  2024-08-15  468.06
15     1016       Unknown  1970-01-01  255.96
16     1017   Credit Card  2024-08-17  240.79
17     1018   Credit Card  2024-08-18  325.22
18     1019       Unknown  2024-08-19  594.30
19     1020   Credit Card  2024-08-20  743.14
20     1021   Credit Card  1970-01

In [13]:
#--------REVIEWS TABLE
reviews_df = reviews_df.fillna(
    {
        'ReviewText':'No Review Submitted!'
    }
)
reviews_df['Rating'] = reviews_df['Rating'].replace('InvalidRating',0)
reviews_df['Rating'] = pd.to_numeric(reviews_df['Rating'])
#print(reviews_df)

print("transform success")
        

transform success


C:\Users\nutan\AppData\Local\Temp\ipykernel_17276\1483356010.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  reviews_df['Rating'] = reviews_df['Rating'].replace('InvalidRating',0)


In [14]:
#load data to database


import mysql.connector

def conn():
    return mysql.connector.connect(
        host = "localhost",
        user = "root",
        password = "root",
        database = "etl_project2"
    )

con = conn()
cursor = con.cursor()

#delete previous records 

#list of tables from which data to be cleared
tables_to_clear = ['payments','reviews','orders','users','products']

#execute sql query in loop
for table in tables_to_clear:
    cursor.execute(f'delete from {table}')
con.commit() # save changes database

#adding records to product table
product_id_mapping = {} # empty dictionary to store the product_id's

for index, row in products_df.iterrows():
    # index - index number given to each row
    # row - hold one row at a time
    # iterrows() - iterate/loop through each row
    cursor.execute('''insert into products(product_name, p_description,product_price,stock_quantity,product_category) 
    values (%s,%s,%s,%s,%s)''',(row['ProductName'],row['Description'],row['Price'], row['StockQuantity'], row['Category']))
    product_id_mapping[index+1] = cursor.lastrowid
    # [index + 1] - key, row index number, lastrowid - valuen- primary key value from db
con.commit()
print(product_id_mapping)

#adding records to user table
user_id_mapping = {}

for index, row in users_df.iterrows():
    cursor.execute('''insert into users(user_name, email, address, u_password) values (%s,%s,%s,%s)''',
                   (row['UserName'],row['Email'],row['Address'],row['Password']))
    user_id_mapping[index+1] = cursor.lastrowid
con.commit()
print(user_id_mapping)
#adding records to orders table

orders_df['UserID'] = orders_df['UserID'].map(user_id_mapping)
#map() will map the new values to each record
print(orders_df['UserID'])
orders_df = orders_df.dropna(subset = 'UserID')
# if any NULL userID delete the row

order_id_mapping = {}
for index, row in orders_df.iterrows():
    cursor.execute(''' insert into orders(userid, order_date, total_amt, email) values (%s,%s,%s,%s)''',
                   (row['UserID'], row['OrderDate'],row['TotalAmount'],row['Email']))
    order_id_mapping[row['OldOrderID']] = cursor.lastrowid
con.commit()
print(order_id_mapping)

#payments table 
payments_df['OrderID'] = payments_df['OrderID'].map(order_id_mapping)
print(payments_df['OrderID'])
payments_df = payments_df.dropna(subset=["OrderID"])

for index, row in payments_df.iterrows():
    cursor.execute('''insert into payments(orderid, payment_method,payment_date, amount) values (%s,%s,%s,%s)''',
                  (row['OrderID'], row['PaymentMethod'], row['PaymentDate'],row['Amount']))
con.commit()

# reviews table
reviews_df['UserID'] = reviews_df['UserID'].map(user_id_mapping)
reviews_df['ProductID'] = reviews_df['ProductID'].map(product_id_mapping)

for index, row in reviews_df.iterrows():
    product_id = product_id_mapping.get(index)
    user_id = user_id_mapping.get(index)
    cursor.execute('''insert into reviews(productid, userid, rating, review_text) values(%s,%s,%s,%s)''',
                  (int(row['ProductID']), int(row['UserID']), row['Rating'], row['ReviewText']))
con.commit()
print("load success")

{1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18, 19: 19, 20: 20, 21: 21, 22: 22, 23: 23, 24: 24, 25: 25, 26: 26, 27: 27, 28: 28, 29: 29, 30: 30, 31: 31, 32: 32, 33: 33, 34: 34, 35: 35, 36: 36, 37: 37, 38: 38, 39: 39, 40: 40, 41: 41, 42: 42, 43: 43, 44: 44, 45: 45, 46: 46, 47: 47, 48: 48, 49: 49, 50: 50, 51: 51, 52: 52, 53: 53, 54: 54, 55: 55, 56: 56, 57: 57, 58: 58, 59: 59, 60: 60}
{1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10}
0     8
1     4
2     1
3     8
4     4
5     6
6     8
7     4
8     3
9     9
10    3
11    9
12    2
13    2
14    2
15    6
16    3
17    9
18    4
19    1
20    4
21    1
22    5
23    4
24    8
25    8
26    7
27    3
28    1
29    1
30    3
31    6
32    7
33    6
34    6
35    6
36    3
37    6
38    8
39    2
40    5
41    1
42    1
43    5
44    3
45    4
46    3
47    1
48    1
49    5
50    6
51    3
52    9
53    5
54    8
55    1
56    5
57    3
58    1
59